In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# Raiz do projeto: ajuste se necessário
RAIZ = Path.cwd()

# Se estiver executando dentro da pasta notebooks/
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent

ARQUIVO = RAIZ / "dados-brutos" / "dados_escolas_liberados.csv"

PASTA_PROCESSADO = RAIZ / "dados" / "processado"
PASTA_RELATORIOS = RAIZ / "docs"

PASTA_PROCESSADO.mkdir(parents=True, exist_ok=True)
PASTA_RELATORIOS.mkdir(parents=True, exist_ok=True)

if not ARQUIVO.exists():
    raise FileNotFoundError(
        f"Arquivo não encontrado: {ARQUIVO}\n"
        "Confira se o CSV está na pasta dados/bruto."
    )

print("Arquivo localizado:", ARQUIVO)

Arquivo localizado: c:\Users\Maria\Downloads\projeto-evasao\dados-brutos\dados_escolas_liberados.csv


In [3]:

# CÉLULA 2 — Ler e inspecionar os dados originais

df = pd.read_csv(
    ARQUIVO,
    sep=None,
    engine="python",
    encoding="utf-8-sig"
)

print("Dimensões do arquivo:", df.shape)

print("\nPrimeiras 5 linhas:")
display(df.head())

print("\nNomes das colunas:")
print(df.columns.tolist())

print("\nTipos de dados:")
display(df.dtypes.rename("tipo").to_frame())

print("\nInformações gerais:")
df.info()


Dimensões do arquivo: (151, 8)

Primeiras 5 linhas:


,TP_DEPENDENCIA,TP_LOCALIZACAO,IN_LABORATORIO_INFORMATICA,IN_BIBLIOTECA,IN_INTERNET,IN_EQUIP_LOUSA_DIGITAL,IN_BANDA_LARGA,QTD_ESCOLAS
0,3,2,NaN,NaN,NaN,NaN,NaN,69436
1,3,1,0.0,0.0,0.0,0.0,NaN,823
2,3,1,1.0,1.0,1.0,1.0,1.0,2691
3,4,1,0.0,1.0,1.0,1.0,1.0,982
4,3,2,1.0,1.0,1.0,0.0,1.0,1183



Nomes das colunas:
['TP_DEPENDENCIA', 'TP_LOCALIZACAO', 'IN_LABORATORIO_INFORMATICA', 'IN_BIBLIOTECA', 'IN_INTERNET', 'IN_EQUIP_LOUSA_DIGITAL', 'IN_BANDA_LARGA', 'QTD_ESCOLAS']

Tipos de dados:


,tipo
TP_DEPENDENCIA,int64
TP_LOCALIZACAO,int64
IN_LABORATORIO_INFORMATICA,float64
IN_BIBLIOTECA,float64
IN_INTERNET,float64
IN_EQUIP_LOUSA_DIGITAL,float64
IN_BANDA_LARGA,float64
QTD_ESCOLAS,int64



Informações gerais:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 151 entries, 0 to 150
Data columns (total 8 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   TP_DEPENDENCIA              151 non-null    int64  
 1   TP_LOCALIZACAO              151 non-null    int64  
 2   IN_LABORATORIO_INFORMATICA  143 non-null    float64
 3   IN_BIBLIOTECA               143 non-null    float64
 4   IN_INTERNET                 143 non-null    float64
 5   IN_EQUIP_LOUSA_DIGITAL      143 non-null    float64
 6   IN_BANDA_LARGA              105 non-null    float64
 7   QTD_ESCOLAS                 151 non-null    int64  
dtypes: float64(5), int64(3)
memory usage: 9.6 KB


In [4]:

# CÉLULA 3 — Verificar qualidade dos dados

# 1. Resumo dos valores ausentes
auditoria = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "valores_ausentes": df.isna().sum(),
    "percentual_ausente": (df.isna().mean() * 100).round(2),
    "valores_distintos": df.nunique(dropna=True)
})

print("=== AUDITORIA DAS COLUNAS ===")
display(auditoria)

# 2. Verificar linhas duplicadas
print("\n=== DUPLICIDADES ===")
print("Linhas duplicadas:", df.duplicated().sum())

# 3. Verificar códigos de dependência e localização
print("\n=== CÓDIGOS DE DEPENDÊNCIA ===")
print(df["TP_DEPENDENCIA"].value_counts(dropna=False).sort_index())

print("\n=== CÓDIGOS DE LOCALIZAÇÃO ===")
print(df["TP_LOCALIZACAO"].value_counts(dropna=False).sort_index())

# 4. Verificar indicadores de infraestrutura
indicadores = [
    "IN_LABORATORIO_INFORMATICA",
    "IN_BIBLIOTECA",
    "IN_INTERNET",
    "IN_EQUIP_LOUSA_DIGITAL",
    "IN_BANDA_LARGA"
]

print("\n=== VALORES DOS INDICADORES ===")
for coluna in indicadores:
    print(f"\n{coluna}")
    print(df[coluna].value_counts(dropna=False).sort_index())

# 5. Verificar quantidade de escolas
print("\n=== QUANTIDADE DE ESCOLAS ===")
display(df["QTD_ESCOLAS"].describe())

print(
    "Registros com quantidade de escolas menor ou igual a zero:",
    (df["QTD_ESCOLAS"] <= 0).sum()
)


=== AUDITORIA DAS COLUNAS ===


,tipo,valores_ausentes,percentual_ausente,valores_distintos
TP_DEPENDENCIA,int64,0,0.00,4
TP_LOCALIZACAO,int64,0,0.00,2
IN_LABORATORIO_INFORMATICA,float64,8,5.30,2
IN_BIBLIOTECA,float64,8,5.30,2
IN_INTERNET,float64,8,5.30,2
IN_EQUIP_LOUSA_DIGITAL,float64,8,5.30,2
IN_BANDA_LARGA,float64,46,30.46,2
QTD_ESCOLAS,int64,0,0.00,121



=== DUPLICIDADES ===
Linhas duplicadas: 0

=== CÓDIGOS DE DEPENDÊNCIA ===
TP_DEPENDENCIA
1    14
2    47
3    50
4    40
Name: count, dtype: int64

=== CÓDIGOS DE LOCALIZAÇÃO ===
TP_LOCALIZACAO
1    84
2    67
Name: count, dtype: int64

=== VALORES DOS INDICADORES ===

IN_LABORATORIO_INFORMATICA
IN_LABORATORIO_INFORMATICA
0.0    70
1.0    73
NaN     8
Name: count, dtype: int64

IN_BIBLIOTECA
IN_BIBLIOTECA
0.0    68
1.0    75
NaN     8
Name: count, dtype: int64

IN_INTERNET
IN_INTERNET
0.0     38
1.0    105
NaN      8
Name: count, dtype: int64

IN_EQUIP_LOUSA_DIGITAL
IN_EQUIP_LOUSA_DIGITAL
0.0    77
1.0    66
NaN     8
Name: count, dtype: int64

IN_BANDA_LARGA
IN_BANDA_LARGA
0.0    48
1.0    57
NaN    46
Name: count, dtype: int64

=== QUANTIDADE DE ESCOLAS ===


count      151.000000
mean      2036.086093
std       7094.775559
min          2.000000
25%         26.500000
50%        161.000000
75%        859.500000
max      69436.000000
Name: QTD_ESCOLAS, dtype: float64

Registros com quantidade de escolas menor ou igual a zero: 0


In [5]:

# CÉLULA 4 — Investigar valores ausentes por categoria

indicadores = [
    "IN_LABORATORIO_INFORMATICA",
    "IN_BIBLIOTECA",
    "IN_INTERNET",
    "IN_EQUIP_LOUSA_DIGITAL",
    "IN_BANDA_LARGA"
]

for coluna in indicadores:
    print(f"\n=== {coluna} ===")

    resumo = (
        df.groupby(
            ["TP_DEPENDENCIA", "TP_LOCALIZACAO"],
            dropna=False
        )
        .agg(
            total_registros=(coluna, "size"),
            valores_ausentes=(coluna, lambda x: x.isna().sum()),
            escolas=("QTD_ESCOLAS", "sum")
        )
        .reset_index()
    )

    resumo["percentual_ausente"] = (
        resumo["valores_ausentes"]
        / resumo["total_registros"] * 100
    ).round(2)

    display(resumo)

# Conferir registros com valores ausentes
print("\n=== LINHAS COM ALGUM INDICADOR AUSENTE ===")

display(
    df.loc[df[indicadores].isna().any(axis=1)]
      .sort_values(["TP_DEPENDENCIA", "TP_LOCALIZACAO"])
)



=== IN_LABORATORIO_INFORMATICA ===


,TP_DEPENDENCIA,TP_LOCALIZACAO,total_registros,valores_ausentes,escolas,percentual_ausente
0,1,1,10,1,705,10.00
1,1,2,4,1,114,25.00
2,2,1,25,1,28935,4.00
3,2,2,22,1,9521,4.55
4,3,1,25,1,73179,4.00
5,3,2,25,1,114733,4.00
6,4,1,24,1,78624,4.17
7,4,2,16,1,1638,6.25



=== IN_BIBLIOTECA ===


,TP_DEPENDENCIA,TP_LOCALIZACAO,total_registros,valores_ausentes,escolas,percentual_ausente
0,1,1,10,1,705,10.00
1,1,2,4,1,114,25.00
2,2,1,25,1,28935,4.00
3,2,2,22,1,9521,4.55
4,3,1,25,1,73179,4.00
5,3,2,25,1,114733,4.00
6,4,1,24,1,78624,4.17
7,4,2,16,1,1638,6.25



=== IN_INTERNET ===


...

,TP_DEPENDENCIA,TP_LOCALIZACAO,total_registros,valores_ausentes,escolas,percentual_ausente
0,1,1,10,1,705,10.00
1,1,2,4,1,114,25.00
2,2,1,25,1,28935,4.00
3,2,2,22,1,9521,4.55
4,3,1,25,1,73179,4.00
5,3,2,25,1,114733,4.00
6,4,1,24,1,78624,4.17
7,4,2,16,1,1638,6.25



=== IN_BANDA_LARGA ===


,TP_DEPENDENCIA,TP_LOCALIZACAO,total_registros,valores_ausentes,escolas,percentual_ausente
0,1,1,10,1,705,10.00
1,1,2,4,1,114,25.00
2,2,1,25,9,28935,36.00
3,2,2,22,6,9521,27.27
4,3,1,25,9,73179,36.00
5,3,2,25,9,114733,36.00
6,4,1,24,8,78624,33.33
7,4,2,16,3,1638,18.75



=== LINHAS COM ALGUM INDICADOR AUSENTE ===


,TP_DEPENDENCIA,TP_LOCALIZACAO,IN_LABORATORIO_INFORMATICA,IN_BIBLIOTECA,IN_INTERNET,IN_EQUIP_LOUSA_DIGITAL,IN_BANDA_LARGA,QTD_ESCOLAS
37,1,1,NaN,NaN,NaN,NaN,NaN,93
63,1,2,NaN,NaN,NaN,NaN,NaN,22
65,2,1,0.0,1.0,0.0,1.0,NaN,4
70,2,1,1.0,1.0,0.0,0.0,NaN,161
77,2,1,1.0,0.0,0.0,1.0,NaN,2
91,2,1,0.0,0.0,0.0,0.0,NaN,241
92,2,1,1.0,0.0,0.0,0.0,NaN,59
108,2,1,NaN,NaN,NaN,NaN,NaN,4587
114,2,1,0.0,1.0,0.0,0.0,NaN,98
116,2,1,1.0,1.0,0.0,1.0,NaN,8


In [6]:

# CÉLULA 5 — Limpeza inicial e exportação

df_limpo = df.copy()

# 1. Padronizar os nomes das colunas
df_limpo.columns = (
    df_limpo.columns
    .str.strip()
    .str.upper()
)

# 2. Garantir que os indicadores sejam numéricos
indicadores = [
    "IN_LABORATORIO_INFORMATICA",
    "IN_BIBLIOTECA",
    "IN_INTERNET",
    "IN_EQUIP_LOUSA_DIGITAL",
    "IN_BANDA_LARGA"
]

for coluna in indicadores:
    df_limpo[coluna] = pd.to_numeric(
        df_limpo[coluna],
        errors="coerce"
    )

# 3. Verificar valores diferentes de 0 e 1
for coluna in indicadores:
    invalidos = (
        df_limpo[coluna].notna()
        & ~df_limpo[coluna].isin([0, 1])
    )

    print(f"{coluna}: {invalidos.sum()} valores inválidos")

# 4. Criar a pasta de saída
PASTA_PROCESSADO = RAIZ / "dados" / "processado"
PASTA_PROCESSADO.mkdir(parents=True, exist_ok=True)

# 5. Salvar a base processada
saida = (
    PASTA_PROCESSADO
    / "dados_escolas_sedap_processados.csv"
)

df_limpo.to_csv(
    saida,
    index=False,
    encoding="utf-8-sig"
)

print("\nBase processada salva em:", saida)
print("Linhas originais:", len(df))
print("Linhas processadas:", len(df_limpo))
print("Colunas processadas:", len(df_limpo.columns))
print("Ausências preservadas:", df_limpo.isna().sum().sum())


IN_LABORATORIO_INFORMATICA: 0 valores inválidos
IN_BIBLIOTECA: 0 valores inválidos
IN_INTERNET: 0 valores inválidos
IN_EQUIP_LOUSA_DIGITAL: 0 valores inválidos
IN_BANDA_LARGA: 0 valores inválidos

Base processada salva em: c:\Users\Maria\Downloads\projeto-evasao\dados\processado\dados_escolas_sedap_processados.csv
Linhas originais: 151
Linhas processadas: 151
Colunas processadas: 8
Ausências preservadas: 78
